<a href="https://colab.research.google.com/github/shettyvishal539-hash/Mini-Project/blob/main/Credit_card_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import joblib

# Load Dataset
data = pd.read_csv("creditcard.csv")

print(data.head())
print(data.info())
print(data['Class'].value_counts())

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=data)
plt.title("Fraud vs Normal Transactions")
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(data['Amount'], bins=50)
plt.title("Transaction Amount Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(data.corr(), cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()

In [ ]:
X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Handle Imbalanced Dataset (SMOTE)

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", pd.Series(y_train_smote).value_counts())

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200)

rf_model.fit(X_train_smote, y_train_smote)

rf_pred = rf_model.predict(X_test)

print("Random Forest Results")
print(classification_report(y_test, rf_pred))

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

sns.heatmap(cm, annot=True, fmt="d")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=10
)

xgb_model.fit(X_train_smote, y_train_smote)

xgb_pred = xgb_model.predict(X_test)

print("XGBoost Results")
print(classification_report(y_test, xgb_pred))

In [ ]:
y_prob = xgb_model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

In [ ]:
ann_model = Sequential()

ann_model.add(Dense(64, activation='relu', input_dim=X_train_smote.shape[1]))
ann_model.add(Dropout(0.3))

ann_model.add(Dense(32, activation='relu'))
ann_model.add(Dropout(0.3))

ann_model.add(Dense(1, activation='sigmoid'))

ann_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = ann_model.fit(
    X_train_smote,
    y_train_smote,
    epochs=10,
    batch_size=256,
    validation_split=0.2
)

In [ ]:
ann_pred = (ann_model.predict(X_test) > 0.5).astype(int)

print("ANN Results")
print(classification_report(y_test, ann_pred))

In [ ]:
importance = rf_model.feature_importances_

feat_importance = pd.Series(importance, index=data.drop("Class", axis=1).columns)

feat_importance.nlargest(10).plot(kind='barh')

plt.title("Top Important Features")
plt.show()

In [ ]:
fraud_prob = xgb_model.predict_proba(X_test)[:,1]

plt.hist(fraud_prob, bins=50)
plt.title("Fraud Probability Distribution")
plt.show()

In [ ]:
joblib.dump(rf_model, "random_forest_model.pkl")
joblib.dump(xgb_model, "xgboost_model.pkl")
ann_model.save("ann_model.h5")

In [ ]:
def predict_transaction(transaction):
    transaction = scaler.transform([transaction])
    prediction = xgb_model.predict(transaction)

    if prediction == 1:
        return "Fraud Transaction"
    else:
        return "Legitimate Transaction"

sample = X.iloc[0].values
print(predict_transaction(sample))